# Thesis Mechanism Spine — living working notebook

**Locked 2026-06-17.** Working notebook for the MSc thesis (defence June 2027).
Commentary + reproducible code for both arms. Edit as the work evolves.

## Thesis claim (falsifiable)
Quantum entanglement improves a biomedical learning task **only when the data carries
matching multi-channel joint structure** (Huang et al. 2021; Kübler et al. 2021).
Tested on two arms sharing one ablation methodology — *remove the entanglement, measure
what is lost.* Extension/application study, not a new method (paper rule #1). Expected
result on the advantage axis is null-or-marginal; the contribution is the **predictive
principle**, not a supremacy claim.

- **Arm 1 — entanglement in the ENCODING** (consolidation of existing per-scheme results).
- **Arm 2 — entanglement in the INITIALIZATION** (new bounded swing; circuit as a fixed
  sampler → no barren plateau). Deciding control = classical-distribution-matched-to-
  quantum-marginals.

Full plan + timeline + kill dates: `thesis_spine_2026.md`. Phase 0 findings:
`arm1_substrate_phase0.md`.

## Arm 1 — Phase 0 substrate

Source of truth: `_perscheme_results.json` (per-scheme, **single run, NO seeds/CI**) and
`_ecg_rawbaseline.json` (ECG raw ROCKET baseline, 5 seeds). All multivariate datasets are
3-channel. Δ_ent = (best entangling single-scheme acc) − (Sep separable acc).

In [ ]:
import json
from pathlib import Path
import pandas as pd

BASE = Path('.')  # run from /Users/eldana/Documents/Quantum/Thesis/QIP
per = json.loads((BASE / '_perscheme_results.json').read_text())

# Entanglement ordering CONFIRMED 2026-06-17:
#   Sep < CRyE < CSE < GBE < PA-CSE < CP-2L
# (PA-CSE = pixel-adaptive CSE; image/ECG only, not in this multivariate set.)
SEP = 'Sep'                                    # separable / no-entanglement baseline
ENTANGLING = ['CRyE', 'CSE', 'GBE', 'CP-2L']   # least → most entangling (present here)

rows = []
for ds, d in per.items():
    s = d['schemes']
    sep_acc = s[SEP]['acc']
    ent = {k: s[k]['acc'] for k in ENTANGLING if k in s}
    best = max(ent, key=ent.get)
    rows.append({
        'dataset': ds,
        'K': d['K'],
        'raw': round(d['raw'], 4),
        'Sep': round(sep_acc, 4),
        'best_ent_scheme': best,
        'best_ent_acc': round(ent[best], 4),
        'delta_ent': round(ent[best] - sep_acc, 4),
        'headroom': round(1 - d['raw'], 4),
        # per-scheme accs in entanglement order, to inspect monotonicity / U-shape
        **{k: round(s[k]['acc'], 4) for k in ENTANGLING if k in s},
    })

df = pd.DataFrame(rows).sort_values('delta_ent', ascending=False).reset_index(drop=True)
df

### Adversarial findings (read before building the metric)

1. **The "entanglement helps UWave, fails ECG" headline is FALSE on canonical splits.**
   On UWave per-scheme, **Sep (separable) is the best single scheme** (0.9125); the
   entangling variants are at or below it (Δ = −0.003). The 0.976 UWave "positive" was a
   cross-scheme **ensemble** on an expanded split — not single-scheme entanglement. Do not
   build the thesis on the ensemble number and call it an entanglement effect.
2. **Every Δ_ent is tiny and has NO error bars** (+0.027 to −0.008, single runs). Plausibly
   within seed noise. **RIGOR GATE:** re-run per-scheme with ≥5 seeds + CIs before asserting
   any entanglement effect.
3. **Confound to break:** the biggest Δ (Handwriting, x/y/force) is also the dataset with the
   most accuracy headroom. "Cross-channel coupling helps" and "there was just room to improve"
   are tangled. Arm 1's job: show a coupling metric *C* predicts Δ_ent **after controlling for
   headroom *H***. If *C* adds nothing over *H*, there is no entanglement-structure result
   (still a finding).

**Revised Arm 1 hypothesis:** Δ_ent = f(cross-channel coupling *C*, headroom *H*). Single-channel
ECG (*C*=0) anchors the low end — the observed entanglement-null becomes a *confirming* point.

In [ ]:
# ECG raw ROCKET baseline (PhysioNet2017, single lead) — zero-coupling anchor.
ecg = json.loads((BASE / '_ecg_rawbaseline.json').read_text())
f1s = [v['macro_f1_nao'] for v in ecg['seeds'].values()]
print(f"ECG raw ROCKET macro-F1: mean={sum(f1s)/len(f1s):.4f}  n_seeds={len(f1s)}")
print(f"config: {ecg['config']}")
# TODO Phase 0: trace ECG QPIE per-scheme numbers out of Phase8_PhysioNet2017_ECG.ipynb
# (memory: Sep/PA-CSE ~0.669/0.670, QMBC-Net -10.7 vs CNN) before citing.

## Open items

- [x] **Scheme entanglement ordering** — CONFIRMED: Sep < CRyE < CSE < GBE < PA-CSE < CP-2L.
- [x] **Image-domain dose-response** — CONFIRMED on PathMNIST (5 seeds, paired t-tests): CSE helps*,
      PA-CSE hurts*. The CSE→PA-CSE within-family sign flip is the keystone. See consolidated cell.
- [ ] Trace ECG QPIE per-scheme exact numbers from `Phase8_PhysioNet2017_ECG.ipynb` (null anchor).
- [ ] **DECISION (open):** demote the multivariate per-scheme set to supporting breadth, OR harden
      it with ≥5 seeds + CIs? Backbone (PathMNIST + ECG) already carries the thesis.
- [ ] (if hardening) define coupling metric *C*; regress Δ on *C* controlling for headroom *H*.

## Arm 2 — initialization (placeholder; build in Phase 1, kill date ~mid-Jul)

6 pre-registered init conditions: He, Xavier, Orthogonal, Entangled-circuit, Product-circuit
(ablation), **Classical-matched-to-quantum-marginals (deciding control)**. Metrics:
epochs-to-threshold, final test acc, loss-landscape sharpness (Hessian-trace), across-seed
variance. Dataset: ECG features (reuse Phase 8 pipeline). Decisive comparison =
entangled vs marginal-matched.

## Open items

- [ ] **Confirm scheme entanglement ordering** (Sep / CRyE / GBE / CP-2L / CSE) — blocks the metric.
- [ ] Trace ECG QPIE per-scheme numbers from `Phase8_PhysioNet2017_ECG.ipynb`.
- [ ] Re-run multivariate per-scheme with ≥5 seeds + CIs (rigor gate).
- [ ] Define coupling metric *C*; regress Δ_ent on *C* controlling for *H*.

## Arm 2 — initialization (placeholder; build in Phase 1, kill date ~mid-Jul)

6 pre-registered init conditions: He, Xavier, Orthogonal, Entangled-circuit, Product-circuit
(ablation), **Classical-matched-to-quantum-marginals (deciding control)**. Metrics:
epochs-to-threshold, final test acc, loss-landscape sharpness (Hessian-trace), across-seed
variance. Dataset: ECG features (reuse Phase 8 pipeline). Decisive comparison =
entangled vs marginal-matched.